# Calcul des indices NDVI, NDWI et MSI sur Sentinel-2

## Connexion GEE

In [1]:
import ee
ee.Authenticate(force=True)
ee.Initialize(project='stress-hydrique-491820')
print("✅ Connexion GEE réussie !")

✅ Connexion GEE réussie !


## Zone d'étude et Collection

In [3]:
zone_tchad = ee.Geometry.Point([17.4, 8.6])

def masquer_nuages(image):
    masque = image.select('SCL').neq(9)
    return image.updateMask(masque)

collection_propre = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(zone_tchad) \
    .filterDate('2023-06-01', '2023-09-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)) \
    .map(masquer_nuages)

print("Images disponibles :", collection_propre.size().getInfo())

Images disponibles : 5


## Calcul NDWI et MSI

### NDWI

In [4]:
premiere_image = collection_propre.first()

# NDWI = (B8 - B11) / (B8 + B11)
ndwi = premiere_image.normalizedDifference(['B8', 'B11']).rename('NDWI')

valeur_ndwi = ndwi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=zone_tchad,
    scale=20  # B11 a une résolution de 20m (pas 10m comme B8/B4)
).getInfo()

print("Valeur NDWI :", valeur_ndwi)

Valeur NDWI : {'NDWI': 0.38086854460093894}


### MSI

In [7]:
msi = premiere_image.select('B11').divide(premiere_image.select('B8')).rename('MSI')

valeur_msi = msi.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=zone_tchad,
    scale=20
).getInfo()

print("Valeur MSI :", valeur_msi)

Valeur MSI : {'MSI': 0.4483637909052274}
